# Tahap 1 — Dataset CXR

**Tujuan:**
1. Memindai secara *read-only* struktur folder `data/raw/kermany` dan `data/raw/rsua` di Google Drive (atau `PROJECT_ROOT` lokal).
2. Menghitung jumlah berkas per folder, kelas, dan ekstensi (memisahkan berkas tersembunyi seperti `.DS_Store`).
3. Mengambil 20 contoh nama berkas per kelas dan menguji ketercakupan pola nama berkas (baik pola Kaggle pada teks proposal maupun pola Mendeley pada dataset aktual).
4. Memeriksa mode warna dan resolusi citra pada sampel acak 200 citra per kelas (`seed=42`) serta mencatat berkas yang gagal dibuka.
5. Menghitung *hash* SHA-256 seluruh berkas citra ke `data/interim/raw_manifest.csv` dan menyimpan ringkasan lengkap ke `data/interim/inventory.json`.

In [ ]:
# Sel 1: Setup lingkungan, mount Drive (bila di Colab), resolusi PROJECT_ROOT, dan cetak versi pustaka
import os
import sys
import json
import time
import random
import hashlib
import platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
from PIL import Image, UnidentifiedImageError

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

# Tambahkan repo root ke sys.path bila src/ tersedia
for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code")]:
    if (candidate / "src" / "paths.py").exists():
        sys.path.insert(0, str(candidate))
        break

try:
    from src.paths import get_project_paths, ensure_output_dirs, load_config
    CFG = load_config()
    PATHS = ensure_output_dirs(cfg=CFG)
    PROJECT_ROOT = PATHS["project_root"]
except Exception:
    CFG = {}
    default_root = "/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia" if IN_COLAB else "./local_mirror"
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", default_root)).expanduser().resolve()
    (PROJECT_ROOT / "data" / "interim").mkdir(parents=True, exist_ok=True)
    PATHS = {
        "project_root": PROJECT_ROOT,
        "data_raw": PROJECT_ROOT / "data" / "raw",
        "kermany_chest_xray": PROJECT_ROOT / "data" / "raw" / "kermany" / "chest_xray",
        "rsua_root": PROJECT_ROOT / "data" / "raw" / "rsua",
        "interim": PROJECT_ROOT / "data" / "interim",
    }

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Python        : {sys.version.split()[0]} ({platform.platform()})")
print(f"NumPy         : {np.__version__}")
print(f"Pillow        : {Image.__version__}")
print(f"IN_COLAB      : {IN_COLAB}")
print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT tidak ditemukan: {PROJECT_ROOT}"

### Analisis Sel 1
Sel menyiapkan eksekusi secara deterministik (`SEED = 42`), mendeteksi apakah notebook berjalan di Google Colab atau lokal, serta memastikan direktori `data/interim` tersedia tanpa memodifikasi isi `data/raw`.

In [ ]:
# Sel 2: Pemindaian pohon folder di bawah PROJECT_ROOT/data/raw
IMAGE_EXTENSIONS = {".jpeg", ".jpg", ".png", ".bmp", ".tif", ".tiff"}
IGNORED_NAMES = {".ds_store", "thumbs.db", "desktop.ini"}

raw_root = PATHS["data_raw"]
assert raw_root.exists(), f"Folder data/raw tidak ditemukan di: {raw_root}"

folder_summary = {}
all_files = []

for dirpath, dirnames, filenames in os.walk(raw_root):
    dirnames.sort()
    filenames.sort()
    rel_dir = Path(dirpath).relative_to(PROJECT_ROOT).as_posix()
    ext_counter = Counter()
    n_images = 0
    n_ignored = 0
    for fn in filenames:
        p = Path(dirpath) / fn
        ext = p.suffix.lower() if p.suffix else "<no_ext>"
        if fn.lower() in IGNORED_NAMES:
            n_ignored += 1
            ext_counter[f"ignored:{fn}"] += 1
            continue
        ext_counter[ext] += 1
        if ext in IMAGE_EXTENSIONS:
            n_images += 1
        all_files.append(p)
    folder_summary[rel_dir] = {
        "n_subdirs": len(dirnames),
        "n_files_valid": len(filenames) - n_ignored,
        "n_images": n_images,
        "n_ignored": n_ignored,
        "extensions": dict(ext_counter),
    }

print(f"Total folder terpindai di data/raw : {len(folder_summary)}")
print(f"Total berkas valid (non-ignored)   : {len(all_files)}")
for rel_d, info in sorted(folder_summary.items()):
    if info["n_files_valid"] > 0 or info["n_ignored"] > 0 or info["n_subdirs"] == 0:
        print(f"  - {rel_d}: valid={info['n_files_valid']} (img={info['n_images']}, ignored={info['n_ignored']}) ext={info['extensions']}")

### Analisis Sel 2
Pemindaian pohon direktori memisahkan berkas citra valid dari berkas sistem seperti `.DS_Store`. Daftar ini menjadi dasar pemeriksaan kontrak struktur folder untuk dataset utama (Kermany) dan dataset verifikasi kualitas mask (RSUA).

In [ ]:
# Sel 3: Inventarisasi rinci Dataset Kermany + uji pola nama berkas + sampel 200 citra per kelas
import re

REGEX_PROPOSAL_PNEUMONIA = re.compile(r"^person(?P<pid>\d+)_(?P<subcls>bacteria|virus)_\d+\.(jpeg|jpg|png)$", re.IGNORECASE)
REGEX_PROPOSAL_NORMAL = re.compile(r"^(?:NORMAL2-)?IM-(?P<pid>\d+)(?:-\d+)?\.(jpeg|jpg|png)$", re.IGNORECASE)
REGEX_MENDELEY = re.compile(r"^(?P<cls>NORMAL|BACTERIA|VIRUS)-(?P<pid>\d+)-(?P<seq>\d+)\.(jpeg|jpg|png)$", re.IGNORECASE)

kermany_root = PATHS["kermany_chest_xray"]
kermany_splits = {}
kermany_by_class = defaultdict(list)
sample_20_filenames = {}
regex_stats = {"proposal_match": 0, "mendeley_match": 0, "unmatched": []}

if kermany_root.exists():
    for split_dir in sorted([d for d in kermany_root.iterdir() if d.is_dir()]):
        split_name = split_dir.name
        kermany_splits[split_name] = {}
        for cls_dir in sorted([d for d in split_dir.iterdir() if d.is_dir()]):
            cls_name = cls_dir.name.upper()
            imgs = sorted([
                p for p in cls_dir.iterdir()
                if p.is_file() and p.name.lower() not in IGNORED_NAMES and p.suffix.lower() in IMAGE_EXTENSIONS
            ])
            kermany_splits[split_name][cls_name] = len(imgs)
            kermany_by_class[cls_name].extend(imgs)
            for p in imgs:
                m_prop = REGEX_PROPOSAL_PNEUMONIA.match(p.name) or REGEX_PROPOSAL_NORMAL.match(p.name)
                m_mend = REGEX_MENDELEY.match(p.name)
                if m_prop:
                    regex_stats["proposal_match"] += 1
                if m_mend:
                    regex_stats["mendeley_match"] += 1
                if not (m_prop or m_mend):
                    regex_stats["unmatched"].append(p.relative_to(PROJECT_ROOT).as_posix())

for cls_name, paths_list in sorted(kermany_by_class.items()):
    sample_20_filenames[cls_name] = [p.name for p in paths_list[:20]]

# Pemeriksaan mode & ukuran pada sampel acak maks 200 citra per kelas
kermany_image_probe = {}
corrupted_files = []

rng = random.Random(SEED)
for cls_name, paths_list in sorted(kermany_by_class.items()):
    k = min(200, len(paths_list))
    sampled = rng.sample(paths_list, k) if k > 0 else []
    modes = Counter()
    sizes = Counter()
    widths, heights = [], []
    for p in sampled:
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                modes[im.mode] += 1
                sizes[f"{im.width}x{im.height}"] += 1
                widths.append(im.width)
                heights.append(im.height)
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            corrupted_files.append({"path": p.relative_to(PROJECT_ROOT).as_posix(), "error": str(exc)})
    kermany_image_probe[cls_name] = {
        "n_total_class": len(paths_list),
        "n_sampled": k,
        "modes": dict(modes),
        "top_10_sizes_wxh": dict(sizes.most_common(10)),
        "width_min_median_max": [int(min(widths)), float(np.median(widths)), int(max(widths))] if widths else [],
        "height_min_median_max": [int(min(heights)), float(np.median(heights)), int(max(heights))] if heights else [],
    }

n_norm = len(kermany_by_class.get("NORMAL", []))
n_pneu = len(kermany_by_class.get("PNEUMONIA", []))
n_tot = n_norm + n_pneu
print("=== RINGKASAN KERMANY ===")
print("Per bagian bawaan :", json.dumps(kermany_splits, indent=2))
print(f"Total Normal      : {n_norm} (Target Proposal Tabel 3.1: 1583) -> {'PASS' if n_norm == 1583 else 'FAIL'}")
print(f"Total Pneumonia   : {n_pneu} (Target Proposal Tabel 3.1: 4273) -> {'PASS' if n_pneu == 4273 else 'FAIL'}")
print(f"Total Keseluruhan : {n_tot} (Target Proposal Tabel 3.1: 5856) -> {'PASS' if n_tot == 5856 else 'FAIL'}")
print(f"Kecocokan Regex   : proposal={regex_stats['proposal_match']}, mendeley={regex_stats['mendeley_match']}, unmatched={len(regex_stats['unmatched'])}")
print("Probe sampel 200  :", json.dumps(kermany_image_probe, indent=2))

### Analisis Sel 3
Sel di atas membandingkan jumlah aktual citra Kermany per kelas terhadap angka terkunci pada Tabel 3.1 proposal (`1.583 Normal + 4.273 Pneumonia = 5.856`), menguji kecocokan pola nama berkas (pola Kaggle vs pola Mendeley), dan memverifikasi integritas berkas serta distribusi mode/ukuran pada sampel 200 citra per kelas.

In [ ]:
# Sel 4: Inventarisasi rinci RSUA (Validated & Annotated) + pemeriksaan ukuran dan pasangan citra-mask
rsua_root = PATHS["rsua_root"]
rsua_summary = {}

if rsua_root.exists():
    for dirpath, dirnames, filenames in os.walk(rsua_root):
        p_dir = Path(dirpath)
        if p_dir.name == "images" and (p_dir.parent / "masks").exists():
            cls_folder = p_dir.parent.name
            subset_label = p_dir.parent.parent.name
            img_files = sorted([p for p in p_dir.iterdir() if p.is_file() and p.name.lower() not in IGNORED_NAMES])
            mask_dir = p_dir.parent / "masks"
            mask_files = sorted([p for p in mask_dir.iterdir() if p.is_file() and p.name.lower() not in IGNORED_NAMES])

            # Cek s.d. 20 sampel untuk ukuran & nilai biner mask
            img_sizes, mask_sizes, mask_unique_vals = Counter(), Counter(), set()
            for im_p, mk_p in list(zip(img_files, mask_files))[:20]:
                try:
                    with Image.open(im_p) as im:
                        img_sizes[f"{im.width}x{im.height}:{im.mode}"] += 1
                    with Image.open(mk_p) as mk:
                        mask_sizes[f"{mk.width}x{mk.height}:{mk.mode}"] += 1
                        arr = np.array(mk)
                        for v in np.unique(arr).tolist()[:10]:
                            mask_unique_vals.add(int(v))
                except Exception as exc:
                    corrupted_files.append({"path": im_p.relative_to(PROJECT_ROOT).as_posix(), "error": str(exc)})

            key = f"{subset_label}/{cls_folder}"
            rsua_summary[key] = {
                "rel_dir": p_dir.parent.relative_to(PROJECT_ROOT).as_posix(),
                "n_images": len(img_files),
                "n_masks": len(mask_files),
                "sample_images": [p.name for p in img_files[:5]],
                "sample_masks": [p.name for p in mask_files[:5]],
                "sampled_image_sizes": dict(img_sizes),
                "sampled_mask_sizes": dict(mask_sizes),
                "sampled_mask_unique_values": sorted(mask_unique_vals),
            }

print("=== RINGKASAN RSUA ===")
print(json.dumps(rsua_summary, indent=2))

### Analisis Sel 4
Sel ini memeriksa kedua bagian RSUA (`Validated` dan `Annotated`), menghitung jumlah pasangan `images/` dan `masks/` per kelas (`Non_Covid` = 32 Normal, `Non_Covid_Pneumonia` = 53 Pneumonia, `Covid` = 207 dikeluarkan pada `Validated`), serta mengecek resolusi `256x256` dan sifat biner *ground truth mask*.

In [ ]:
# Sel 5: Hitung SHA-256 seluruh berkas citra & simpan raw_manifest.csv + inventory.json
import csv

COMPUTE_SHA256 = True  # Set True untuk menghitung sha256 seluruh berkas di data/raw

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

t0 = time.time()
manifest_rows = []
sha_counter = Counter()

for p in all_files:
    rel_p = p.relative_to(PROJECT_ROOT).as_posix()
    ext = p.suffix.lower()
    size_bytes = p.stat().st_size
    digest = sha256_file(p) if (COMPUTE_SHA256 and ext in IMAGE_EXTENSIONS) else ""
    if digest:
        sha_counter[digest] += 1
    manifest_rows.append({
        "rel_path": rel_p,
        "filename": p.name,
        "ext": ext,
        "size_bytes": size_bytes,
        "sha256": digest,
    })

manifest_csv_path = PATHS["interim"] / "raw_manifest.csv"
with open(manifest_csv_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["rel_path", "filename", "ext", "size_bytes", "sha256"])
    writer.writeheader()
    writer.writerows(manifest_rows)

n_dup_byte_groups = sum(1 for _, c in sha_counter.items() if c > 1)
n_dup_byte_extra = sum(c - 1 for _, c in sha_counter.items() if c > 1)

val_norm = rsua_summary.get("Data Chest X-Ray RSUA (Validated)/Non_Covid", {}).get("n_images", 0)
val_pneu = rsua_summary.get("Data Chest X-Ray RSUA (Validated)/Non_Covid_Pneumonia", {}).get("n_images", 0)
val_covid = rsua_summary.get("Data Chest X-Ray RSUA (Validated)/Covid", {}).get("n_images", 0)

proposal_checks = {
    "kermany_normal_eq_1583": {"actual": n_norm, "expected": 1583, "status": "PASS" if n_norm == 1583 else "FAIL"},
    "kermany_pneumonia_eq_4273": {"actual": n_pneu, "expected": 4273, "status": "PASS" if n_pneu == 4273 else "FAIL"},
    "kermany_total_eq_5856": {"actual": n_tot, "expected": 5856, "status": "PASS" if n_tot == 5856 else "FAIL"},
    "rsua_validated_normal_eq_32": {"actual": val_norm, "expected": 32, "status": "PASS" if val_norm == 32 else "FAIL"},
    "rsua_validated_pneumonia_eq_53": {"actual": val_pneu, "expected": 53, "status": "PASS" if val_pneu == 53 else "FAIL"},
    "rsua_validated_covid_excluded_eq_207": {"actual": val_covid, "expected": 207, "status": "PASS" if val_covid == 207 else "FAIL"},
}

inventory = {
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "seed": SEED,
    "proposal_checks": proposal_checks,
    "kermany": {
        "splits": kermany_splits,
        "n_normal": n_norm,
        "n_pneumonia": n_pneu,
        "n_total": n_tot,
        "regex_match_counts": {
            "proposal_kaggle_pattern": regex_stats["proposal_match"],
            "mendeley_pattern": regex_stats["mendeley_match"],
            "unmatched_count": len(regex_stats["unmatched"]),
            "unmatched_examples": regex_stats["unmatched"][:20],
        },
        "sample_20_filenames_per_class": sample_20_filenames,
        "sample_200_probe_per_class": kermany_image_probe,
    },
    "rsua": rsua_summary,
    "other_folders": {
        "kermany_manual_masks_count": len(list((PROJECT_ROOT / "data/raw/kermany_manual_masks").glob("*"))) if (PROJECT_ROOT / "data/raw/kermany_manual_masks").exists() else 0,
        "seg_public_count": len(list((PROJECT_ROOT / "data/raw/seg_public").glob("*"))) if (PROJECT_ROOT / "data/raw/seg_public").exists() else 0,
        "unet_frozen_models": [p.name for p in (PROJECT_ROOT / "models/unet_frozen").glob("*")] if (PROJECT_ROOT / "models/unet_frozen").exists() else [],
    },
    "corrupted_files": corrupted_files,
    "raw_manifest": {
        "csv_path": manifest_csv_path.relative_to(PROJECT_ROOT).as_posix(),
        "total_rows": len(manifest_rows),
        "duplicate_sha256_groups_across_all_raw": n_dup_byte_groups,
        "duplicate_sha256_extra_files_across_all_raw": n_dup_byte_extra,
        "elapsed_sec": round(time.time() - t0, 2),
    },
    "folder_tree_summary": folder_summary,
}

inventory_json_path = PATHS["interim"] / "inventory.json"
with open(inventory_json_path, "w", encoding="utf-8") as fh:
    json.dump(inventory, fh, indent=2, ensure_ascii=False)

print(f"Selesai dalam {inventory['raw_manifest']['elapsed_sec']} dtk.")
print(f"Tersimpan: {inventory_json_path}")
print(f"Tersimpan: {manifest_csv_path}")
print("\n=== PROPOSAL CHECKS ===")
print(json.dumps(proposal_checks, indent=2))